In [ ]:
import polars as pl
from pathlib import Path
import geopandas as gpd
import h3
from google.colab import drive
from shapely.geometry import Point
import ee
import numpy as np
from shapely import STRtree
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor


In [ ]:
drive.mount('/content/drive')

In [ ]:
base = Path('/content/drive/MyDrive')

In [ ]:
df = pl.read_csv(base / 'renewably_wind' / 'final_df_processed.csv')
df.describe()

In [ ]:
def explode_to_children(df: pl.DataFrame, target_res: int) -> pl.DataFrame:
    """Split each H3 cell into children at target_res."""
    rows = []
    for h3_index in df["h3_index"].to_list():
        for child in h3.cell_to_children(h3_index, res=target_res):
            lat, lng = h3.cell_to_latlng(child)
            rows.append({
                "h3_index": child,
                "parent_h3": h3_index,
                "lat": lat,
                "lng": lng,
            })
    return pl.DataFrame(rows)

df = explode_to_children(df, 9)
df.head()

## Wind turbine geojson

In [ ]:
turbine_gdf = gpd.read_file(base / 'renewably_wind' / 'wind_us_farms.geojson')

In [ ]:
turbine_gdf = turbine_gdf.to_crs(epsg=4326)
centroids = turbine_gdf.geometry.to_crs(3857).centroid.to_crs(4326)
turbine_gdf['longitude'] = centroids.x
turbine_gdf['latitude'] = centroids.y


In [ ]:
turbine_gdf = turbine_gdf[['fid', 'longitude', 'latitude']]
turbine_gdf.shape

In [ ]:
res = 9

turbine_cells = (
    pl.from_records(
        [
            (h3.latlng_to_cell(lat, lon, res),)
            for lat, lon in zip(
                turbine_gdf["latitude"],
                turbine_gdf["longitude"],
            )
        ],
        schema=["h3_index"],
        orient="row"
    )
    .unique()
)


In [ ]:
df = df.join(
    turbine_cells.with_columns(pl.lit(True).alias("has_turbine")),
    on="h3_index",
    how="left",
).with_columns(pl.col("has_turbine").fill_null(False))

In [ ]:
df.head()

In [ ]:
df['has_turbine'].value_counts()


In [ ]:
n = df['has_turbine'].value_counts().filter(~pl.col("has_turbine"))["count"].item() * .07

true_df = df.filter(pl.col("has_turbine"))
false_df = df.filter(~pl.col("has_turbine")).sample(n=n, seed=42)

df_balanced = pl.concat([true_df, false_df])

In [ ]:
df_balanced.head()
df_balanced['has_turbine'].value_counts()


In [ ]:
df_balanced.head()

### Download Data Sets

In [ ]:
ee.Authenticate()
ee.Initialize(project='renewably')

In [ ]:
gdf = gpd.GeoDataFrame(
    df_balanced, 
    geometry=gpd.points_from_xy(df_balanced["lng"], df_balanced["lat"]),
    crs="EPSG:4326"
)

gdf.to_csv(base / 'renewably_wind' / 'points.csv', index=False)

In [ ]:
# Code to generate terrain features on earth engine

points = ee.FeatureCollection('projects/renewably/assets/points')

# Sources
nlcd = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD').first()
nlcd_landcover = nlcd.select(['landcover'])
nlcd_impervious = nlcd.select(['impervious'])
elevation = ee.Image('USGS/SRTMGL1_003').select(['elevation'])
slope_img = ee.Terrain.slope(elevation).rename('slope')

# ERA5 monthly mean — 10m wind components
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
    .filterDate('2020-01-01', '2023-12-31') \
    .select(['u_component_of_wind_10m', 'v_component_of_wind_10m'])

# Compute wind speed magnitude, then average across all months
def wind_speed(img):
    return img.expression(
        'sqrt(u**2 + v**2)',
        {'u': img.select('u_component_of_wind_10m'),
         'v': img.select('v_component_of_wind_10m')}
    ).rename('wind_speed')

mean_wind = era5.map(wind_speed).mean()

# OpenLandMap soil taxonomy great groups
soil = ee.Image('OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01') \
    .select(['grtgroup']).rename('soil_type')

# Stack into a single multi-band image
stacked = (nlcd_landcover
    .addBands(nlcd_impervious)
    .addBands(elevation)
    .addBands(slope_img)
    .addBands(mean_wind)
    .addBands(soil)
)


# Single reduceRegions call — each band gets its own column
results = stacked.reduceRegions(
    collection=points,
    reducer=ee.Reducer.first(),
    scale=30
)

task = ee.batch.Export.table.toAsset(
    collection=results,
    description='terrain_features_all5',
    assetId='projects/renewably/assets/terrain_features_all5'
)
task.start()

In [ ]:
earthengine_df = pl.read_csv(base / 'renewably_wind' / 'terrain_features_final.csv')
earthengine_df.head()


In [ ]:
# Rename for your functions
earthengine_df = earthengine_df.rename({
    '0': 'h3_index',
    '1': 'parent_h3',
    '2': 'lat',
    '3': 'lng',
    '4': 'has_turbine',
    'landcover': 'land_type',
    'elevation': 'elevation_m',
    'slope': 'slope_deg'
})

earthengine_df = earthengine_df.drop(['system:index', '.geo', 'parent_h3'])

In [ ]:
earthengine_df.describe()

In [ ]:
# Fix nulls
earthengine_df = earthengine_df.with_columns([
    pl.col("elevation_m").cast(pl.Float64).fill_null(0),
    pl.col("slope_deg").cast(pl.Float64).fill_null(0),
    pl.col("land_type").fill_null("0"),  # "0" = no data / unclassified
    pl.col("impervious").fill_null("0"),
    pl.col("soil_type").fill_null("0"),
])

In [ ]:
# Encodings 

# NLCD Land Cover — group into meaningful categories for turbine siting
landcover_encoding = {
    "0":  0,   # No data / unclassified
    "11": 1,   # Open water
    "12": 1,   # Perennial ice/snow
    "21": 2,   # Developed, open space
    "22": 3,   # Developed, low intensity
    "23": 4,   # Developed, medium intensity
    "24": 5,   # Developed, high intensity
    "31": 6,   # Barren land
    "41": 7,   # Deciduous forest
    "42": 8,   # Evergreen forest
    "43": 9,   # Mixed forest
    "52": 10,  # Shrub/scrub
    "71": 11,  # Grassland/herbaceous
    "81": 12,  # Pasture/hay
    "82": 13,  # Cultivated crops
    "90": 14,  # Woody wetlands
    "95": 15,  # Emergent herbaceous wetlands
}

# Impervious  encoding
impervious_encoding = {
    "0":  0,   # Unclassified
    "20": 1,   # Primary road
    "21": 2,   # Secondary road
    "22": 3,   # Tertiary road
    "23": 4,   # Thinned road
    "24": 5,   # Non-road non-energy impervious
    "25": 6,   # Microsoft buildings
    "26": 7,   # LCMAP impervious
    "27": 0,   # Wind turbines → mask to unclassified (LEAKAGE)
    "28": 8,   # Well pads
    "29": 9,   # Other energy production
}

earthengine_df = earthengine_df.with_columns(
    pl.col("impervious")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace(impervious_encoding, default=0),
    pl.col("land_type")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace(landcover_encoding, default=0)
)


In [ ]:
earthengine_df.describe()

In [ ]:
# Build code-to-order mapping from the class table
soil_code_to_order = {
    0: 0,  # NODATA
    # Alfisols (-alfs)
    1: 1, 2: 1, 4: 1, 6: 1, 7: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1,
    14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 25: 1, 26: 1, 27: 1,
    28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 38: 1, 39: 1, 41: 1, 42: 1,
    43: 1, 44: 1, 45: 1, 46: 1,
    # Andisols (-ands)
    50: 2, 58: 2, 59: 2, 61: 2, 63: 2, 64: 2, 74: 2, 75: 2, 76: 2,
    77: 2, 80: 2,
    # Aridisols (-ids)
    82: 3, 83: 3, 85: 3, 86: 3, 87: 3, 89: 3, 90: 3, 92: 3, 93: 3,
    94: 3, 96: 3, 97: 3, 98: 3, 99: 3, 100: 3, 101: 3, 102: 3, 103: 3,
    104: 3, 105: 3, 107: 3, 110: 3, 111: 3, 112: 3, 113: 3, 114: 3,
    115: 3, 116: 3,
    # Entisols (-ents)
    118: 4, 119: 4, 120: 4, 121: 4, 122: 4, 123: 4, 124: 4, 126: 4,
    131: 4, 133: 4, 134: 4, 135: 4, 136: 4, 137: 4, 138: 4, 139: 4,
    140: 4, 141: 4, 142: 4, 143: 4, 144: 4, 145: 4, 146: 4, 147: 4,
    148: 4, 149: 4, 153: 4, 154: 4, 155: 4,
    # Histosols (-ists)
    179: 5, 180: 5, 181: 5, 182: 5, 183: 5, 184: 5, 185: 5, 186: 5,
    189: 5, 190: 5, 191: 5, 196: 5, 201: 5, 202: 5, 203: 5,
    # Inceptisols (-epts)
    206: 6, 207: 6, 208: 6, 209: 6, 210: 6, 212: 6, 213: 6, 215: 6,
    216: 6, 217: 6, 218: 6, 219: 6, 220: 6, 221: 6, 222: 6, 225: 6,
    226: 6, 228: 6, 229: 6, 230: 6, 231: 6, 233: 6, 234: 6, 235: 6,
    245: 6, 246: 6, 247: 6, 248: 6, 249: 6, 250: 6, 251: 6, 252: 6,
    253: 6, 254: 6, 255: 6, 256: 6,
    # Mollisols (-olls)
    268: 7, 269: 7, 270: 7, 271: 7, 272: 7, 273: 7, 274: 7, 275: 7,
    276: 7, 277: 7, 278: 7, 279: 7, 280: 7, 283: 7, 284: 7, 285: 7,
    286: 7, 287: 7, 289: 7, 290: 7, 291: 7, 292: 7, 294: 7, 296: 7,
    297: 7, 298: 7, 299: 7, 300: 7, 301: 7, 302: 7, 303: 7, 306: 7,
    307: 7, 308: 7, 309: 7, 310: 7, 311: 7, 312: 7,
    # Spodosols (-ods)
    342: 8, 343: 8, 345: 8, 348: 8, 349: 8, 350: 8, 351: 8, 353: 8,
    354: 8, 356: 8, 357: 8, 358: 8, 367: 8, 368: 8,
    # Ultisols (-ults)
    370: 9, 371: 9, 372: 9, 373: 9, 374: 9, 375: 9, 376: 9, 377: 9,
    378: 9, 381: 9, 385: 9, 387: 9, 388: 9, 389: 9, 390: 9, 391: 9,
    396: 9, 399: 9, 401: 9,
    # Vertisols (-erts)
    403: 10, 405: 10, 406: 10, 409: 10, 410: 10, 412: 10, 413: 10,
    414: 10, 415: 10, 417: 10, 418: 10, 419: 10, 420: 10, 422: 10,
    424: 10, 429: 10, 430: 10, 431: 10, 432: 10, 433: 10,
}

earthengine_df = earthengine_df.with_columns(
    pl.col("soil_type")
      .cast(pl.Int32)
      .replace_strict(soil_code_to_order, default=0)
      .cast(pl.UInt8)
)

In [ ]:
earthengine_df.describe()

In [ ]:
# Change var name
df = earthengine_df

In [ ]:
df.shape

In [93]:
def proximity_to_transmission_line(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate the distance to the nearest transmission line for each point."""
    transmission_lines = gpd.read_file(base / 'renewably_wind' / 'us_transmission_lines.geojson')
    transmission_lines = transmission_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(transmission_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("transmission_line_dist_km", distances_km)
    )



def proximity_to_road(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate distance to nearest road for each point"""
    road_lines = gpd.read_file(base / 'renewably_wind' / 'tl_2023_us_primaryroads.shp').set_crs(epsg=4269)
    road_lines = road_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(road_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("road_dist_km", distances_km)
    )


def proximity_to_residential(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate distance to nearest residential area for each point"""
    return

def endangered_species(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate the endangered species for each point"""
    return

def native_land(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate the native land for each point"""
    return

def proximity_to_airport(df: pl.DataFrame) -> pl.DataFrame:
    airports = gpd.read_file(base / 'renewably_wind' / 'airports.geojson')
    airports = airports.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(airports.geometry.values)
    
    _, distances_m = tree.query_nearest(points.geometry.values, return_distance=True)

    distances_km = distances_m / 1000.0

    return df.with_columns(
        pl.Series("airport_dist_km", distances_km)
    )

def wind_speed(df: pl.DataFrame) -> pl.DataFrame:
    wind_speeds = pd.read_csv(base / 'renewably_wind' / 'us_wind_speed_dataset_2022.csv')

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(wind_speeds[["lat", "lon"]], wind_speeds["annual_mean_wind_speed"])

    pdf = df.to_pandas()
    for col in ["wind_speed", "annual_mean_wind_speed"]:
        if col in pdf.columns:
            pdf = pdf.drop(columns=[col])

    pdf["wind_speed"] = knn.predict(pdf[["lat", "lng"]].rename(columns={"lng": "lon"}))

    return pl.from_pandas(pdf)
    

In [85]:
df = pl.read_csv(base / 'renewably_wind' / 'final_df_processed_with_proximity.csv')

In [ ]:
df = proximity_to_road(df)

In [ ]:
df = proximity_to_transmission_line(df)

In [91]:
df = wind_speed(df)

In [94]:
df = proximity_to_airport(df)

In [102]:
df.sample(10)

h3_index,lat,lng,has_turbine,elevation_m,impervious,land_type,slope_deg,soil_type,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
str,f64,f64,bool,f64,i64,i64,f64,i64,f64,f64,f64,f64
"""89276ba0197ffff""",43.858913,-82.760012,false,220.0,0,4,1.286093,1,97.517615,0.744837,5.18647,17.787708
"""892765a8423ffff""",47.803424,-82.282011,false,373.0,0,0,0.0,6,213.576046,212.777475,2.475755,108.371788
"""8948105260bffff""",24.66865,-104.343455,false,2094.0,0,0,16.938373,7,579.385256,539.385523,2.464874,511.577348
"""8926093a033ffff""",40.0435,-94.047468,false,234.0,0,13,2.592721,7,3.997527,4.883394,3.419458,12.543535
"""8926f11286bffff""",37.872771,-99.617418,true,713.0,0,13,0.92741,7,116.62577,5.918351,4.278444,19.105087
"""8926c3d50d7ffff""",35.22613,-99.116953,false,535.0,0,10,1.465757,6,23.953581,3.222789,4.54046,14.376387
"""8926daa33c3ffff""",32.867083,-101.160526,true,815.0,0,13,4.309407,10,56.79122,0.926617,4.560248,11.95721
"""8926cb2f68bffff""",31.782192,-96.763118,true,171.0,0,12,2.151068,10,31.29114,3.091611,3.832583,4.910701
"""8926c6a0943ffff""",36.568879,-99.291609,true,663.0,0,11,10.321642,7,122.691378,0.62584,4.411615,12.787411


In [103]:
df.write_csv(base / 'renewably_wind' / 'final_df_processed_with_proximity.csv')

## Modeling Time

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

X = df[['elevation_m', 'slope_deg', 'land_type', 'impervious', 'soil_type', 'road_dist_km', 'transmission_line_dist_km', 'wind_speed', 'airport_dist_km']]

y = df['has_turbine']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])

pipeline.fit(X_train, y_train)



h3_index,lat,lng,has_turbine,elevation_m,impervious,land_type,slope_deg,soil_type,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
str,f64,f64,bool,f64,i64,i64,f64,i64,f64,f64,f64,f64
"""892a8e5aecfffff""",38.065782,-80.494997,true,1314.0,0,7,14.954538,9,20.211045,2.652109,2.069863,23.306144
"""8926b014543ffff""",42.89936,-109.243753,false,3289.0,0,10,9.89352,6,137.907373,38.908333,2.415497,42.901702
"""892930b037bffff""",30.113259,-122.251885,false,0.0,0,0,0.0,0,521.730383,512.40721,1.772118,436.72686
"""89440a7b517ffff""",28.557155,-85.109326,false,0.0,0,0,0.0,0,224.18616,125.161968,2.862597,121.88862
"""892a56954bbffff""",35.921758,-69.13284,false,0.0,0,0,0.0,0,597.973354,568.57567,2.883218,569.039892
"""8928c22d09bffff""",48.270113,-124.518925,false,223.0,0,8,17.566302,6,150.502338,4.870036,1.387327,12.541751
"""892793b4abbffff""",46.570451,-109.728297,true,1388.0,0,11,3.270851,7,82.893605,0.39826,2.904605,15.869046
"""8948522ca23ffff""",30.494126,-111.7338,false,418.0,0,0,2.143653,3,119.576426,122.923208,3.128139,132.928448
"""8929a16ed2fffff""",35.054724,-118.392585,true,1516.0,0,11,2.92591,7,21.912131,0.737524,1.722291,5.876652
